In [248]:
import csv
from pydantic import BaseModel
from ollama import chat
from typing import Literal

In [249]:
DATA_FOLDER = "../data/"
CSV_PREDICATE_ACTOR = f"{DATA_FOLDER}predicate_actor.csv"
CSV_ACTORS_MAPPING = f"{DATA_FOLDER}actors_mapping.csv"
CSV_PREDICATE_ACTOR = f"{DATA_FOLDER}predicate_actor.csv"
TXT_SENTENCES_FOLDER = f"{DATA_FOLDER}txt_sentences/"
LIST_OF_YEARS = ["1996", "1997", "1998", "1999", "2000", "2001", "2002", "2003", "2005", "2006", "2007", "2024"]

In [250]:
SYSTEM_PROMPT = """
You extract structured negotiation propositions from a sentence.

Return one proposition per actor when relevant.

Definitions:
- actor: must be one of the actors provided by the user
- predicate: copy the predicate exactly as it appears in the sentence text. Do not lemmatize it, normalize it, or paraphrase it.
- negotiation_point: copy the shortest exact text span from the sentence that captures the issue, proposal, policy, amendment, or text being supported, opposed, reported, or otherwise referred to. Do not paraphrase it.
- predicate_type:
    - "support" if the actor supports, proposes, requests, favors, backs, endorses, or accepts the negotiation point
    - "opposition" if the actor opposes, rejects, objects to, or is described as opposing the negotiation point
    - "other" if the sentence does not express a clear support or opposition stance

Important rules:
1. Use only actors from the provided actor list.
2. Do not invent actors.
3. Return one proposition per actor when the sentence provides enough evidence.
4. If several actors share the same position, return one proposition per actor.
5. If multiple actors are coordinated and share the same predicate, return one proposition per actor with the same predicate and negotiation_point.
6. If an actor is described as "opposed by X, Y and Z", then X, Y and Z should each get predicate_type = "opposition".
7. If an actor makes a proposal, classify it as "support".
8. If the sentence contains a nominal predicate such as "proposal by X", "request by X", or "objection by X", extract that nominal predicate and classify it according to its meaning.
9. If a reporting verb introduces a clearer support or opposition predicate, use the support/opposition predicate instead of the reporting verb.
10. The predicate must be copied exactly from the sentence.
11. The negotiation_point must be copied from the sentence, not rewritten.
12. Return only valid JSON matching the schema.
"""

In [251]:
class Proposition(BaseModel):
    actor: str
    predicate: str
    negotiation_point: str
    predicate_type: Literal["support", "opposition", "other"]


class PropositionList(BaseModel):
    propositions: list[Proposition]

In [252]:
def get_actors_in_sentence(sentence: str) -> set:
    temp_actors = set()
    actors = set()

    # Fill temp_actors with actors_mapping.csv
    with open(CSV_ACTORS_MAPPING, mode ='r')as file:
        csvFile = csv.reader(file)
        next(csvFile, None)  # skip the headers
        for lines in csvFile:
            if lines[0] in sentence or lines[1] in sentence:
                if lines[0].upper() == "CHINA" and "G-77/CHINA" in sentence:
                    continue
                if lines[0].upper() == "G-77" and "G-77/CHINA" in sentence:
                    continue
                temp_actors.add(lines[1])

    # Cleaning and mapping
    for actor in temp_actors:
        with open(CSV_ACTORS_MAPPING, mode ='r')as file:
            csvFile = csv.reader(file)
            for lines in csvFile:
                actor = actor.upper()
                if lines[0].upper() == actor or lines[1].upper() == actor:
                    actors.add(lines[1])

    return actors

In [253]:
def extract_predicate_in_sentence(sentence: str) -> Proposition:
    response = chat(
        model = 'ministral-3:14b',
        format = Proposition.model_json_schema(),
        messages = [
                {
                    "role": "system",
                    "content": "Extract the actor, predicate, negotiation point, and predicate type from the following sentence. The predicate type should be either 'reporting' if the predicate is a reporting verb, 'stance' if the predicate is a stance word, or 'other' if it is neither."
                },
                {
                    "role": "user",
                    "content": sentence
                },
            ],
        )

    return Proposition.model_validate_json(response.message.content)

In [254]:
def extract_predicate_in_sentence(sentence: str, sentence_actors: list[str]) -> PropositionList:
    system_prompt = SYSTEM_PROMPT
    user_prompt = f"Sentence: {sentence} Available actors: {sentence_actors}"

    response = chat(
        model="ministral-3:8b",
        format=PropositionList.model_json_schema(),
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
    )

    return PropositionList.model_validate_json(response.message.content)

In [255]:
def build_csv_predicate_actor(year: str, sentence_id: str, propositions: list[Proposition]):
    with open(CSV_PREDICATE_ACTOR, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        for proposition in propositions:
            writer.writerow([year, sentence_id, proposition.actor, proposition.predicate, proposition.negotiation_point, proposition.predicate_type])


In [256]:
def build_predicate_for_year(year: str):
    i = 0
    with open(f"{TXT_SENTENCES_FOLDER}{year}.txt", mode='r', encoding="utf-8") as sentences_file:
        txtFile = csv.reader(sentences_file)
        for line in txtFile:
            sentence_id = i
            sentence = line[0]
            sentence_actors = get_actors_in_sentence(sentence)
            proposition = extract_predicate_in_sentence(sentence, sentence_actors)
            build_csv_predicate_actor(year, sentence_id, proposition.propositions)
            i += 1


In [257]:
with open(CSV_PREDICATE_ACTOR, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(['year', 'sentence_id', 'actor', 'predicate', 'negotiation_point', 'predicate_type'])


for year in LIST_OF_YEARS:
    print(f"Processing year {year}...")
    build_predicate_for_year(year)
    print(f"Finished processing year {year}.")


Processing year 1996...
Finished processing year 1996.
Processing year 1997...
Finished processing year 1997.
Processing year 1998...
Finished processing year 1998.
Processing year 1999...
Finished processing year 1999.
Processing year 2000...
Finished processing year 2000.
Processing year 2001...
Finished processing year 2001.
Processing year 2002...
Finished processing year 2002.
Processing year 2003...
Finished processing year 2003.
Processing year 2005...
Finished processing year 2005.
Processing year 2006...
Finished processing year 2006.
Processing year 2007...
Finished processing year 2007.
Processing year 2024...
Finished processing year 2024.
